# 27 · 交付：让别人五分钟看懂这个项目

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → [27 交付 ✅]
```

> **这一章的代码在合并请求 #27 里**（和第 25、26 章同一个）。运行这本笔记不花钱。

## 背景：做完了，别人看不懂，等于没做

前面 26 章做了一个罕见病诊断 agent：工具、agent、评测、防泄露、服务、和别人比、读病历。东西很多。

但看这个项目的人——招聘的人、面试官、未来的同事——**通常只花五分钟**：README 第一屏、一段视频、一页结果。五分钟里看不出"这是什么、好在哪、靠不靠得住"，后面的 26 章就没人看了。

**公司里的做法**：项目做完要"交付"，不是把代码一扔。常见的几样东西：
- **架构图**：一张图说清楚各部分怎么连
- **演示**：一段录屏，几十秒看完整个流程
- **结果报告**：一页纸，主要结论、数字、置信区间、局限
- **README**：仓库的门面，第一屏决定别人还看不看
- **复盘**：做对了什么、没做的是什么、下一步

这一章把这几样做出来。

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 给谁看 |
|---|---|---|---|
| 1 | 架构图 | Mermaid：用文字画图，和代码放在一起，GitHub 直接画出来 | 想知道"怎么搭的"的工程师 |
| 2 | 按罕见程度拆开看 | Orphanet 患病率分级；事后分析，只描述 | 想知道"价值在哪"的人 |
| 3 | 演示录屏 | Playwright 自动录：可以随时重录 | 所有人 |
| 4 | 结果报告页 | 一个网页，所有考试一页看完 | 招聘方、面试官 |
| 5 | README 第一屏 | 视频、一句话、一张表 | 打开仓库的人 |
| 6 | 求职材料 | 项目汇总、简历要点、面试故事 | 你自己 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 架构图

### 在干什么
画一张图：病历进来，经过哪些部分，变成诊断出去；评测和记录在哪。

### 背景
前面每一章都讲了一个部分。但没有一张图把它们放在一起，别人得读完 26 章才拼得出全貌。

### 用了什么技术，为什么
**Mermaid**：用几十行文字描述"哪个框连哪个框"，GitHub、GitLab、很多文档工具都会把它直接画成图。好处：
- 图和代码放在同一个仓库里，改代码时顺手改图，**不会过时**（画在 PPT 里的图，三个月后就和代码对不上了）
- 改动能在合并请求里逐行审查
- 不需要任何绘图软件

图在 README 里（"How the parts fit"）。写好以后，我用 Mermaid 11 在无头浏览器里真的画了一遍、看了截图，改了三次：第一版太宽；第二版箭头指进框里，Mermaid 就不按左右排了；第三版把"评测用的是同一个 agent"写在框里，不画箭头。

```mermaid
flowchart TB
    note["Clinical note, free text"] --> reader
    subgraph reader["Note reader"]
        direction LR
        r1["LLM lists each finding<br/>with the note's words<br/>and its status"] --> r2["HPO term search<br/>Qdrant hybrid<br/>BGE + BM25"]
        r2 --> r3["LLM picks a term<br/>among the candidates only"]
        r4["Dictionary pass<br/>HPO names in the note"] --> r3
    end
    reader --> check["A clinician checks the terms"]
    check --> api
    coded["Or HPO terms directly<br/>+ candidate genes"] --> api
    subgraph api["Diagnosis service (FastAPI)"]
        direction LR
        f["No-model ranking first<br/>the fallback"] --> agent["Agent<br/>LangChain create_agent<br/>deepseek-flash, ≤ 8 calls"]
        agent <--> tools["Tools, also over MCP<br/>rank_diseases · check_match<br/>lookup_disease · find_disease<br/>find_similar_patients<br/>rank_candidate_genes"]
        tools --- kb[("Pinned knowledge<br/>HPO 2026-09-01<br/>6,014 published patients")]
    end
    api --> out["Up to ten diseases, each with<br/>its reason, evidence and sources"]
    api -.->|traces| mlflow[("MLflow<br/>traces · evaluation runs")]
    harness["Evaluation harness · mlflow.genai.evaluate<br/>runs this same agent, each case's paper masked<br/>leak audit on every tool output"] --> mlflow
```

（在 GitHub 上看 README，这一段会画成图。）

### 公司里怎么做
**架构即代码**（diagrams as code）：Mermaid、PlantUML、C4 模型。技术评审、新人入职、对外介绍，第一页都是这张图。

---

## 第 2 步 · 按罕见程度拆开看

### 在干什么
把留出集的 840 道题，按确诊的病"有多罕见"分成几组，分别看各系统的第 1 名命中率。

### 背景
"罕见病"差别很大：有的每十万人里就有一个（比如半乳糖血症），医学书上都有；有的全世界只报道过几个家系。一个自然的问题：**agent 的本事到底用在哪？** 是常见的罕见病，还是极罕见的？

### 用了什么技术，为什么
- **Orphanet 的患病率分级**（en_product9_prev.xml，和第 18 章用的 Orphanet 同一个版本，CC BY 4.0），按第 05 章的规矩登记、校验
- 每个病取 Orphanet 已核实的**时点患病率**（全球优先），没有就用出生患病率、终生患病率、年发病率；只知道"报道过几个家系"的，算"不到百万分之一"
- OMIM 编号和 Orphanet 只用**完全对应**（exact mapping），和第 18 章一样

写代码时出过一个错，值得记一下：我按印象把"出生患病率"写成了 `Birth prevalence`，文件里其实叫 `Prevalence at birth`。结果半乳糖血症、Holt-Oram 这种常见的病都"没有分级"。**一个明显不对的结果（大家都知道的病查不到）就是信号**，回去看文件里的原文，才发现名字写错了。

**这是事后分析**：方案里没有事先写这个问题，所以只描述现象，不做检验、不下结论。

### 执行

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path
from raredx.data import cases as cs
from raredx.evals.scoring import wilson
from raredx.knowledge import prevalence as P
band = P.omim_bands(Path("data/raw/orphanet-prevalence/2026-06-23/en_product9_prev.xml"),
                    Path("data/raw/orphanet/2026-06-23/en_product1.xml"))
disease = {c.id: c.disease_id for c in cs.read(Path("data/processed/phenopacket-store/0.1.27"))}
rows = list(csv.DictReader(open("docs/comparison_hits.tsv"), delimiter="\t"))
groups = defaultdict(list)
for r in rows:
    groups[band.get(disease[r["case_id"]], "unknown")].append(r)
systems = {"agent": "最终 agent", "fixed-pipeline": "固定流程", "lirical": "LIRICAL", "llm-direct": "模型直接猜"}
print(f"{'罕见程度':<22}{'题数':>5}{'病数':>5}" + "".join(f"{v:>14}" for v in systems.values()))
for b in P.BANDS:
    g = groups[b]
    cells = "".join(f"{sum(r[s + '@1'] == '1' for r in g) / len(g):>15.1%}" for s in systems)
    print(f"{b:<24}{len(g):>5}{len({disease[r['case_id']] for r in g}):>5}{cells}")

### 结果怎么看

| 罕见程度 | 题数（病数） | 最终 agent | 固定流程 | LIRICAL | 模型直接猜 |
|---|---|---|---|---|---|
| 十万分之一以上 | 42（25） | 57.1% | 50.0% | 40.5% | **61.9%** |
| 百万分之一到九 | 66（12） | **80.3%** | 74.2% | 50.0% | 53.0% |
| 不到百万分之一 | 272（115） | **52.6%** | 45.6% | 40.1% | **9.6%** |
| Orphanet 没有分级 | 460（327） | 29.6% | 25.0% | 20.4% | 7.2% |

- **相对常见的罕见病，模型自己就知道**：直接猜 61.9%，和 agent 差不多
- **越罕见，模型越不知道**：不到百万分之一的病，直接猜只对 9.6%；agent 用工具查，对 52.6%
- 所以 **agent 的价值在"长尾"**：模型没见过、记不住的那些病。这正是罕见病诊断最难、最需要帮助的地方
- "没有分级"的 327 个病里，290 个在 Orphanet 里根本没有对应的条目；这一组里四分之一的题，遮住病人自己的论文以后按规则就答不出来

**但要说清楚**：第一组只有 25 个病，同一个病的病人会抱团（一个病答对，它的几个病人一起对）。所以这是"看到一个现象"，不是"证明了一件事"。

### 公司里怎么做
**分层分析**（segment analysis）：总体指标会掩盖差异，要按用户群、按场景拆开看，才知道产品的价值在哪、短板在哪。同时要分清**预先计划的分析**和**事后探索**：后者只能提出假设，下一轮评测再去验证。

### 面试怎么说
> "Split by Orphanet prevalence, the bare model knows the relatively common rare diseases — 62% at first place, level with the agent — but only 10% of those rarer than one in a million, where the agent reaches 53%. The agent's value is in the long tail. That was post hoc, so I report it as descriptive."

---

## 第 3 步 · 演示录屏

### 在干什么
录一段视频：贴一段病历 → 读出词条（在病历上标出原话）→ 勾选 → 诊断 → 确诊的病排第一 → 看证据。

### 背景
别人没有你的电脑、你的密钥、你的数据，跑不起来这个项目。一段一分钟的视频，是让别人"亲眼看到它能用"最快的办法。

### 用了什么技术，为什么
**Playwright**（微软的浏览器自动化工具）：写一个脚本控制真的浏览器，一步一步点页面，同时录屏。`scripts/record_demo.py`。

为什么不手动录：
- **能重录**：代码改了、页面变了，跑一次脚本就是新视频。手录的视频，改一个字就得重来
- **每次一样**：同样的步骤、同样的停顿，不会手抖、不会点错
- Playwright 本来是公司里做**端到端测试**（E2E test）的标准工具，录屏是它顺带的功能

录出来是 MP4（2 MB 左右，分享用），再用 ffmpeg 转一个 GIF（3 MB 左右，README 里直接播放）。

**踩过的一个坑**：第一次录出来的视频里，半乳糖血症的病历中间混进了另一个病人的病历。原因是脚本点了"选这个病人"以后，页面还没换好就去复制文本框的内容，复制到的是上一个病人的。修法：先等文本框里真的出现这个病人的病历（"galactose"），再复制。**自动化脚本最常见的错，就是没等页面准备好。** 每次录完我都抽了几帧截图看过。

### 执行
视频在 `docs/img/demo-note.gif`（README 第一屏）和 `docs/img/demo-note.mp4`。在笔记里直接看：

In [ ]:
from IPython.display import Image
Image(filename="docs/img/demo-note.gif")

自己重录（要先开服务和页面，录一次约一美分）：

```
RAREDX_SERVICE_READ_NOTES=true uv run raredx serve &
uv run raredx demo &
uv run --group record playwright install chromium     # 第一次
uv run --group record python scripts/record_demo.py docs/img/demo-note
```

### 公司里怎么做
产品演示、发布说明里的动图，很多就是这样自动录的；和 E2E 测试共用一套脚本，界面一改，测试和演示一起更新。

---

## 第 4 步 · 结果报告页

### 在干什么
做一个网页，把所有考试的结果放在一页上：和外部系统比、和 DeepRare 比、840 道全量、按罕见程度、读病历、花费和速度。

报告页：**https://claude.ai/artifact/8nTB1vZPtxu5yCm7Wcjhn9**（默认只有你能打开；要给别人看，在页面右上角的 Share 里分享）

### 背景
README 是给打开仓库的工程师看的，很长，也很技术。招聘方、面试官想要的是：**一页，几分钟，看清楚这个系统多好、和谁比、靠不靠得住。**

### 用了什么技术，为什么
- 页面是一个**仪表盘**（dashboard）：每个图、每个表、每个数字都标明来自哪一份数据，读者点一下就能看到出处
- 六份数据由 `scripts/report_data.py` 算出来，来源只有两种：仓库里的逐题结果表（`docs/comparison_hits.tsv`、`docs/comparison_llm_hits.tsv` 等），和 MLflow 里的考试记录。**页面上没有一个数字是手抄的**
- 把临时脚本搬进仓库时，我重写了一遍，再跑一次，和已经上传的六份数据逐个比对：完全一样
- 页面上照样写了"读病历的预注册结论没成立"，用红色标签标出来

### 公司里怎么做
数字都能追溯到数据，叫**可追溯的报告**（lineage / provenance）。公司里的周报、实验报告，评审第一个问题往往是"这个数从哪来的"。答得上来，报告才有人信。

---

## 第 5 步 · README 第一屏

### 在干什么
重写 README 的开头：打开仓库第一眼看到什么。

### 背景
README 是仓库的门面。现在的 README 很完整，但第一屏是一大段文字。看的人要先知道"这是什么、好在哪"，细节再往下看。

### 用了什么技术，为什么
第一屏放三样：
1. **一句话**：这是什么
2. **一段动图**：贴病历 → 出诊断
3. **一张表**：主要结果，带置信区间，和最强的几个对手比

往下才是架构图、每一场考试的细节、怎么运行。

### 公司里怎么做
好的开源项目 README 都是这个结构：一句话、一张图或动图、快速开始、再往下是细节。叫"**倒金字塔**"：最重要的放最前面。

---

## 第 6 步 · 求职材料

### 在干什么
把这个项目变成求职用的材料：项目汇总、简历要点、面试故事。

### 背景
你的求职汇总文档（`/Users/pipi/Jobhunt/资料/项目汇总_求职用.md`）原来是按"全部做完以后的预期样子"写的。现在做完了，要改成**实际做出来的样子**：哪些做了、数字是多少、哪些没做、为什么。

### 做了什么
- **项目一整个重写**：最前面加了"一页看完：结果"；每一项标了状态（✅ 做完 · ❌ 没做、为什么 · 📋 可以接着做）；读病历改成"agent 之前的单独一步"（原来写成了 agent 的工具），并写上没成立的那条结论；文字检索写成"做了、量了、删了"
- 最后加了一节"面试时可以讲的故事"，七个
- 项目二没动。旧版备份在临时文件夹里

### 简历要点（英文，可以直接用）

- Built a rare-disease diagnosis agent (LangChain/LangGraph on DeepSeek, tools also served over MCP, FastAPI, MLflow, Docker). On 200 held-out published patients it ranks the confirmed disease first in 43%, against 33% for GPT-6.1 asked directly and 27.5% for DeepRare (Nature 2026) on the same model, paired p < 0.01, at about 13 s and $0.006 per case.
- Designed a leakage-controlled evaluation on MLflow GenAI evaluation: each case's source paper masked from every tool with an automated leak audit, pre-registered plans, Wilson intervals and paired sign tests; showed that unmasked benchmarks on published cases inflate scores by over 30 points.
- Added clinical-note reading: the LLM extracts findings with their status and picks HPO codes from candidates found by hybrid Qdrant retrieval, never writing an ID itself. Evaluated it on public annotated corpora and reported the failed pre-registered claim (tied a dictionary on clinical notes). Patient text is kept out of logs and traces by an MLflow span processor, with tests.
- Shipped it as a service: no-model fallback, streamed steps, concurrency limits and deadlines, a CI-built Docker image smoke-tested in CI, and a Playwright-recorded demo.

### 面试怎么说
最常被问的三个问题，答案都在汇总文档第 12 节，每个都能落到一个具体数字上：
1. "直接用最强的模型不行吗？"
2. "你怎么知道分数是真的？"（防泄露）
3. "有什么没做成的？"（读病历那条，以及为什么照实报告）

---

## 这个项目做完了什么，没做什么

| | 状态 |
|---|---|
| agent：工具、预算、按时交卷、出错能改 | ✅ |
| 评测：留出集、预注册、防泄露审计、逐题配对检验 | ✅ |
| 对照：传统方法、固定流程、模型直接猜、LIRICAL、三家前沿模型、DeepRare | ✅ |
| 有基因检测结果的考法 | ✅（模拟的测序结果） |
| 服务：API、流式、兜底、Docker、MCP | ✅ |
| 读病历：公开语料上评测、病历不留底 | ✅ |
| 演示、架构图、报告页 | ✅ |
| 真实的外显子测序数据 | ❌ 没有公开的、带确诊的数据 |
| 真实医院的病历上验证 | ❌ 需要伦理审批和数据协议 |
| 用 agent 的记录训练一个小模型 | 📋 下一步可以做 |

## 一句话

**做完一个项目，最后一步是让别人五分钟看懂它，而且看到的每个数字都经得起追问。**